In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
with open(r"C:\Projects\Transformers\Combined.txt","r",encoding="utf-8") as T:
    T = T.read(500000)

chars = sorted(set(T))
vocab = len(chars)
stoi = {c: i for i, c in enumerate(chars)}  
itos = {i: c for i, c in enumerate(chars)} 
def encode(s):
    return [stoi[c] for c in s]
def decode(indices):
    return ''.join([itos[i] for i in indices])
text = encode(T)

In [38]:
seq_len = 1
batch = 1
def get_batch():
    max_start = len(text) - seq_len - 1
    ix = torch.randint(0,max_start , (batch,))

    x = torch.stack([
        torch.tensor(text[i:i+seq_len], dtype=torch.long)
        for i in ix
    ])
    
    y = torch.stack([
        torch.tensor(text[i+1:i+seq_len+1], dtype=torch.long)
        for i in ix
    ])


    x = x.squeeze(0)
    y = y.squeeze(0)
    return x, y
x,y = get_batch()
print(x,y)
class Bigram(nn.Module):
    def __init__(self, vocab):
        super().__init__()
        self.emb = nn.Embedding(vocab, 512)
        self.mlp = nn.Linear(512,vocab)
    def forward(self, x):
        x = self.emb(x)
        x = self.mlp(x)
        return x


model = Bigram(vocab)
optimizer = torch.optim.AdamW(model.parameters(),lr=1e-4)
loss_type = nn.CrossEntropyLoss()

tensor([72]) tensor([55])


In [39]:
for i in range(8000):
    x, y = get_batch()

    logits = model(x)          
    loss = loss_type(logits, y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if i % 100 == 0:
        print(f"Step {i}, Loss: {loss.item()}")


Step 0, Loss: 8.425699234008789
Step 100, Loss: 7.035299777984619
Step 200, Loss: 5.907556533813477
Step 300, Loss: 3.0992891788482666
Step 400, Loss: 5.546184539794922
Step 500, Loss: 6.810957908630371
Step 600, Loss: 7.131406784057617
Step 700, Loss: 5.23674201965332
Step 800, Loss: 6.17888069152832
Step 900, Loss: 3.0934553146362305
Step 1000, Loss: 1.64788818359375
Step 1100, Loss: 3.9911177158355713
Step 1200, Loss: 2.2080025672912598
Step 1300, Loss: 8.026078224182129
Step 1400, Loss: 2.3488709926605225
Step 1500, Loss: 2.2908551692962646
Step 1600, Loss: 4.759912490844727
Step 1700, Loss: 10.032203674316406
Step 1800, Loss: 5.374538421630859
Step 1900, Loss: 4.166534423828125
Step 2000, Loss: 2.2676737308502197
Step 2100, Loss: 3.8273210525512695
Step 2200, Loss: 3.6987781524658203
Step 2300, Loss: 2.226822853088379
Step 2400, Loss: 3.9150781631469727
Step 2500, Loss: 2.8749890327453613
Step 2600, Loss: 4.051100254058838
Step 2700, Loss: 1.2856837511062622
Step 2800, Loss: 3.256

In [43]:
@torch.no_grad()
def generate(model, start_token, max_new_tokens=25, temperature=1.0):
    model.eval()

    idx = torch.tensor([start_token], dtype=torch.long)

    for _ in range(max_new_tokens):
        idx_cond = idx[:, -seq_len:]
        logits = model(idx_cond)
        logits = logits[:, -1, :]
         
        logits = logits / temperature

        probs = F.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)

        idx = torch.cat([idx, next_token], dim=1)

    return idx

start_char = 'To'
start_token = encode(start_char) 

out = generate(model, start_token, max_new_tokens=300, temperature=0.7)
print(decode(out[0].tolist()))


Tonthin ilid pe pe s he tat d hereled shin s an d thee ikbust blld ne bller buse or an d fonll iome t p. g at sher s calise wat bs thend the hithe blen was cou ater rde t ber is tind d the thmeus whisthowithicu rve ange cheree thwhond d hecd con t t atiche t wand hid his t an hey t temus t nd.
plle t 
